In [ ]:
# Long short Term memory(LSTM)
import math
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense, LSTM
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

In [ ]:
# Download adjusted closes through the historical forecast target.
import yfinance as yf

ticker = 'AAPL'
forecast_date = pd.Timestamp('2019-12-17')
download_end = (forecast_date + pd.Timedelta(days=1)).date().isoformat()
quotes = yf.download(
    ticker, start='2012-01-01', end=download_end, auto_adjust=True,
    multi_level_index=False,
).sort_index()
if quotes.empty or forecast_date not in quotes.index:
    raise ValueError(f'No {ticker} quote for {forecast_date.date()}.')
if not quotes.index.is_unique or not np.isfinite(quotes['Close']).all():
    raise ValueError('Expected one finite adjusted close per date.')

df = quotes.loc[quotes.index < forecast_date]
df.head(10)

In [ ]:
# Get the number of rows and columns
df.shape

In [ ]:
# Visualize the closing price history
plt.figure(figsize=(16, 8))
plt.title('AAPL adjusted close history')
plt.plot(df['Close'])
plt.xlabel('Date', fontsize=18)
plt.ylabel('Adjusted close (USD)', fontsize=18)
plt.show()

In [ ]:
# Split the pre-forecast history chronologically.
lookback = 60
data = df['Close']
dataset = data.to_numpy().reshape(-1, 1)
train_data_len = math.ceil(len(dataset) * 0.8)
if not lookback < train_data_len < len(dataset):
    raise ValueError('Need 60 prior closes and at least one target in each split.')
train_data_len

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail(10)

In [ ]:
# Learn scaling from the training split only.
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(dataset[:train_data_len])
scaled_data = scaler.transform(dataset)
scaled_data

In [ ]:
# Create training windows and next-session targets.
train_data = scaled_data[:train_data_len, :]
x_train = []
y_train = []
for i in range(lookback, len(train_data)):
    x_train.append(train_data[i - lookback:i, 0])
    y_train.append(train_data[i, 0])

In [ ]:
# Convert the x_train and y_train to numpy array
x_train, y_train = np.array(x_train), np.array(y_train)

In [ ]:
# Reshape the data
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))
x_train.shape

In [ ]:
# Biuld the LSTM model
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))

In [ ]:
# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
# Train the model
model.fit(x_train, y_train, batch_size=1, epochs=1)

In [ ]:
# Use the preceding 60 observed closes for each held-out target date.
target_rows = range(train_data_len, len(data))
x_test = np.array([
    scaled_data[i - lookback:i] for i in target_rows
])
test_dates = data.index[train_data_len:]

In [ ]:
# Pair each prediction with its target date and adjusted actual close.
predictions = scaler.inverse_transform(model.predict(x_test)).ravel()
if predictions.shape != (len(test_dates),) or not np.isfinite(predictions).all():
    raise ValueError('Expected one finite prediction per held-out date.')
valid = data.loc[test_dates].to_frame('Actual')
valid['Predicted'] = predictions

In [ ]:
# Compute RMSE in adjusted US dollars for the matched holdout dates.
errors = valid['Predicted'] - valid['Actual']
rmse = np.sqrt(np.mean(errors ** 2))
rmse

In [ ]:
# Plot training history and the dated holdout comparison.
train = data.iloc[:train_data_len]
plt.figure(figsize=(16, 8))
plt.title('AAPL one-session-ahead holdout predictions')
plt.xlabel('Date', fontsize=18)
plt.ylabel('Adjusted close (USD)', fontsize=18)
plt.plot(train, label='Train')
plt.plot(valid['Actual'], label='Actual')
plt.plot(valid['Predicted'], label='Predicted')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Show the actual and predicted adjusted closes by target date.
valid

In [ ]:
# Predict the reserved target using only closes before that date.
forecast_window = scaled_data[-lookback:].reshape(1, lookback, 1)
predicted_close = scaler.inverse_transform(model.predict(forecast_window)).item()
actual_close = quotes.at[forecast_date, 'Close']
pd.DataFrame(
    {'Actual': [actual_close], 'Predicted': [predicted_close]},
    index=pd.DatetimeIndex([forecast_date], name='Date'),
)